# Notebook 08 — CP03: Epidemic Model Implementation

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 8 of 12  
**Time estimate:** 75 minutes

> CP03 is a clean, documented version of the epidemic model comparison
> from Module 15 MP01. The code here has NumPy-style docstrings, is
> self-contained, and is designed for the portfolio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

rng = np.random.default_rng(42)

# ---- Shared parameters ----
N    = 1000   # population
BETA = 0.3    # transmission rate
GAMMA= 0.1    # recovery rate
I0   = 5      # initial infected
R0_THEORY = BETA / GAMMA
N_REPLICATES = 20
T_MAX = 100.0

print(f'Parameters: N={N}, β={BETA}, γ={GAMMA}, R₀={R0_THEORY:.1f}')
print(f'Herd immunity threshold: {(1 - 1/R0_THEORY):.1%}')

# ---- Model 1: ODE SIR ----

def sir_ode(t, y, beta, gamma, N):
    """SIR model ODE right-hand side.

    Parameters
    ----------
    t : float
        Current time.
    y : array-like of shape (3,)
        State [S, I, R].
    beta : float
        Transmission rate (contacts per unit time × transmission probability).
    gamma : float
        Recovery rate (1/mean infectious period).
    N : int
        Total population (constant).

    Returns
    -------
    list of float
        Derivatives [dS/dt, dI/dt, dR/dt].
    """
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

t_eval = np.linspace(0, T_MAX, 300)
sol = solve_ivp(sir_ode, [0, T_MAX], [N-I0, I0, 0],
                 args=(BETA, GAMMA, N), t_eval=t_eval, dense_output=True)
S_ode, I_ode, R_ode = sol.y

# ---- Model 2: Network SIR (Erdos-Renyi) ----

def build_er_graph(n, p, rng):
    """Build Erdos-Renyi random graph as adjacency list."""
    adj = {i: [] for i in range(n)}
    for i in range(n):
        for j in range(i+1, n):
            if rng.random() < p:
                adj[i].append(j); adj[j].append(i)
    return adj

def network_sir(adj, N, beta, gamma, i0, rng_sim):
    """Stochastic network SIR simulation."""
    state = np.zeros(N, dtype=int)  # 0=S, 1=I, 2=R
    infected_init = rng_sim.choice(N, i0, replace=False)
    state[infected_init] = 1
    t, S_list, I_list, R_list = 0.0, [], [], []
    dt = 1.0
    while t < T_MAX and (state == 1).any():
        S_list.append((state==0).sum()); I_list.append((state==1).sum()); R_list.append((state==2).sum())
        new_state = state.copy()
        for node in range(N):
            if state[node] == 0:
                n_inf = sum(1 for nb in adj.get(node,[]) if state[nb]==1)
                if n_inf > 0 and rng_sim.random() < 1-(1-beta)**n_inf:
                    new_state[node] = 1
            elif state[node] == 1:
                if rng_sim.random() < gamma:
                    new_state[node] = 2
        state = new_state; t += dt
    S_list.append((state==0).sum()); I_list.append((state==1).sum()); R_list.append((state==2).sum())
    return np.array(S_list), np.array(I_list), np.array(R_list)

# ---- Model 3: Spatial ABM ----

def spatial_sir(N, beta, gamma, i0, grid_size, rng_sim):
    """Spatial ABM SIR: agents on a 2D grid, infection from Moore neighbourhood."""
    state  = np.zeros(N, dtype=int)  # 0=S, 1=I, 2=R
    pos    = rng_sim.integers(0, grid_size, size=(N, 2))
    state[rng_sim.choice(N, i0, replace=False)] = 1
    S_list, I_list, R_list = [], [], []
    for _ in range(int(T_MAX)):
        S_list.append((state==0).sum()); I_list.append((state==1).sum()); R_list.append((state==2).sum())
        new_state = state.copy()
        # Random walk
        moves = rng_sim.integers(-1, 2, size=(N, 2))
        pos = np.clip(pos + moves, 0, grid_size - 1)
        for i in range(N):
            if state[i] == 0:
                dist = np.abs(pos - pos[i]).max(axis=1)  # Chebyshev distance
                neighbors_inf = ((dist <= 1) & (state == 1) & (np.arange(N) != i)).sum()
                if neighbors_inf > 0 and rng_sim.random() < 1-(1-beta)**neighbors_inf:
                    new_state[i] = 1
            elif state[i] == 1:
                if rng_sim.random() < gamma:
                    new_state[i] = 2
        state = new_state
    S_list.append((state==0).sum()); I_list.append((state==1).sum()); R_list.append((state==2).sum())
    return np.array(S_list), np.array(I_list), np.array(R_list)

# Run stochastic models (20 replicates each)
print(f'Building ER graph (N={N}, mean degree ≈ {int(N*0.02)})...')
MEAN_DEGREE = 12; p_er = MEAN_DEGREE / (N - 1)
adj = build_er_graph(N, p_er, rng)

net_I_curves, spatial_I_curves = [], []
for rep in range(N_REPLICATES):
    rng_rep = np.random.default_rng(rep)
    _, I_net, _ = network_sir(adj, N, BETA, GAMMA, I0, rng_rep)
    net_I_curves.append(I_net)
    rng_rep2 = np.random.default_rng(rep + 100)
    _, I_sp, _  = spatial_sir(N, BETA*0.6, GAMMA, I0, 50, rng_rep2)
    spatial_I_curves.append(I_sp)

print(f'Replicates done: network SIR={len(net_I_curves)}, spatial ABM={len(spatial_I_curves)}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

t_ode = t_eval

def pad_curves(curves):
    max_len = max(len(c) for c in curves)
    return np.array([np.pad(c, (0, max_len - len(c)), mode='edge') for c in curves])

net_I_mat  = pad_curves(net_I_curves)
sp_I_mat   = pad_curves(spatial_I_curves)
t_net  = np.arange(net_I_mat.shape[1])
t_sp   = np.arange(sp_I_mat.shape[1])

# Panel A: Epidemic curves
ax = axes[0, 0]
ax.plot(t_ode, I_ode, 'k-', lw=2.5, label='ODE SIR (deterministic)')
ax.plot(t_net, net_I_mat.mean(axis=0), color='#4e79a7', lw=2, label='Network SIR (ER)')
ax.fill_between(t_net, net_I_mat.mean(axis=0)-net_I_mat.std(axis=0),
                  net_I_mat.mean(axis=0)+net_I_mat.std(axis=0), color='#4e79a7', alpha=0.25)
ax.plot(t_sp, sp_I_mat.mean(axis=0), color='#e15759', lw=2, label='Spatial ABM')
ax.fill_between(t_sp, sp_I_mat.mean(axis=0)-sp_I_mat.std(axis=0),
                  sp_I_mat.mean(axis=0)+sp_I_mat.std(axis=0), color='#e15759', alpha=0.25)
ax.set_xlabel('Time (days)'); ax.set_ylabel('Infected (I)')
ax.set_title('A. Epidemic curves (mean ± SD,\nn=20 replicates for stochastic models)')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel B: Final size comparison
ax = axes[0, 1]
final_size_net = [net_I_curves[i][-1] + (N - sum([net_I_curves[i][-1]])) for i in range(N_REPLICATES)]
# R(t=inf) = N - S(t=inf)
final_r_net = [N - net_I_curves[i][-1] - sum(1 for x in [] if False) for i in range(N_REPLICATES)]
# Simpler: just use last time point R
final_size_net = np.array([N - curves[-1] for curves in [net_I_curves[i] for i in range(N_REPLICATES)]])
final_size_sp  = np.array([N - curves[-1] for curves in [spatial_I_curves[i] for i in range(N_REPLICATES)]])
final_size_ode = N - S_ode[-1]
positions = [1, 2, 3]
data = [np.repeat(final_size_ode, N_REPLICATES), final_size_net, final_size_sp]
colors_f = ['black', '#4e79a7', '#e15759']
for i, (d, color, pos) in enumerate(zip(data, colors_f, positions)):
    parts = ax.violinplot([d], positions=[pos], widths=0.6)
    for pc in parts['bodies']: pc.set_facecolor(color); pc.set_alpha(0.7)
ax.set_xticks([1,2,3]); ax.set_xticklabels(['ODE SIR', 'Network\nSIR', 'Spatial\nABM'])
ax.set_ylabel('Final epidemic size (total infected)')
ax.set_title('B. Final epidemic size comparison\n(n=20 replicates)')
ax.text(-0.12, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel C: Peak infected
ax = axes[1, 0]
peak_net = np.array([curves.max() for curves in net_I_curves])
peak_sp  = np.array([curves.max() for curves in spatial_I_curves])
peak_ode = I_ode.max()
for i, (d, color, pos, label) in enumerate(zip(
    [np.repeat(peak_ode, N_REPLICATES), peak_net, peak_sp],
    colors_f, positions,
    ['ODE SIR', 'Network SIR', 'Spatial ABM']
)):
    ax.boxplot([d], positions=[pos], widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=color, alpha=0.5))
ax.set_xticks([1,2,3]); ax.set_xticklabels(['ODE SIR', 'Network\nSIR', 'Spatial\nABM'])
ax.set_ylabel('Peak infected (max I)')
ax.set_title('C. Peak infected comparison\n(boxplot, n=20 replicates)')
ax.text(-0.12, 1.05, 'C', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel D: Summary statistics table
ax = axes[1, 1]
ax.axis('off')
col_labels = ['Model', 'Peak I (mean)', 'Final size (mean)', 'R₀ (theory/data)']
row_data = [
    ['ODE SIR', f'{peak_ode:.0f}', f'{final_size_ode:.0f}', f'{R0_THEORY:.1f}'],
    ['Network SIR', f'{peak_net.mean():.0f}±{peak_net.std():.0f}', f'{final_size_net.mean():.0f}±{final_size_net.std():.0f}', 'N/A'],
    ['Spatial ABM', f'{peak_sp.mean():.0f}±{peak_sp.std():.0f}', f'{final_size_sp.mean():.0f}±{final_size_sp.std():.0f}', 'N/A'],
]
table = ax.table(cellText=row_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1, 2)
ax.set_title('D. Summary statistics table')
ax.text(-0.12, 1.05, 'D', transform=ax.transAxes, fontsize=12, fontweight='bold')

plt.suptitle('Module 20 CP03: Epidemic Model Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp03_models.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Why does the spatial ABM produce a smaller and slower epidemic than the
> ODE SIR model with the same β and γ? What biological scenario does
> this better represent — a respiratory virus in a well-mixed population,
> or a sexually transmitted infection in a contact network?]*

---
*Next: `09_cp03_parameter_estimation.ipynb`*